In [ ]:
import sys
import time
import numpy as np
from pathlib import Path
from scipy.optimize import minimize

import numpy as np
import time
from pytket import Circuit
from guppylang import guppy
from guppylang.std.quantum import qubit, measure, measure_array
from guppylang.std.builtins import result, array, comptime
from scipy.optimize import minimize

In [ ]:
# ============================================================
# STEP 1: Load the insurance problem and get classical baseline
# ============================================================
# Adjust this path to your repo structure
sys.path.insert(0, ".../yquantum_challenge/Travelers-LTIMindTree-Quantinuum-Challenge/code_examples")

#from the sources folder, import relevant functions
from src.insurance_model import (
    load_ltm_instance, subsample_problem, solve_ilp,
    get_ilp_matrices, build_ilp,
)

# Load full instance (reads the 6 CSV files in the YQH26_data folder and assembles them into 
# a single Python object (BundlingProblem) that contains everything about the insurance optimization problem

from pathlib import Path
data_path = Path(".../yquantum_challenge/Travelers-LTIMindTree-Quantinuum-Challenge/data/YQH26_data")
ltm_full = load_ltm_instance(data_path)

In [ ]:
# Start with the smallest instance: 5 coverages x 2 packages = 10 binary vars
problem = subsample_problem(ltm_full, n_coverages=5, n_packages=2)
print(f"Problem: {problem.N} coverages x {problem.M} packages = {problem.N * problem.M} binary vars")

In [ ]:
# Build QUBO Matrix

def build_qubo(c_vec, A_mat, b_vec, senses, lam):
    n = len(c_vec)
    Q = np.zeros((n, n))
    for i in range(n):
        Q[i, i] -= c_vec[i]
    for k in range(len(b_vec)):
        a_row = A_mat[k]
        b_k = b_vec[k]
        for i in range(n):
            if a_row[i] == 0:
                continue
            Q[i, i] += lam * (a_row[i]**2 - 2 * b_k * a_row[i])
            for j in range(i+1, n):
                if a_row[j] == 0:
                    continue
                Q[i, j] += lam * 2 * a_row[i] * a_row[j]
    Q = Q + np.triu(Q, 1).T
    return Q

In [ ]:
# Extract Terms from the QUBO Matrix

def extract_qubo_terms(Q):
    n = Q.shape[0]
    diag = [(i, float(Q[i,i])) for i in range(n) if Q[i,i] != 0]
    off_diag = [(i, j, float(Q[i,j])) 
                for i in range(n) for j in range(i+1, n) if Q[i,j] != 0]
    return diag, off_diag

In [ ]:
# Build the QAOA Circuit iwth Pytket

def build_qaoa_pytket(Q, n_qubits, gammas, betas):
    diag, off_diag = extract_qubo_terms(Q)
    depth = len(gammas)
    circ = Circuit(n_qubits)
    for i in range(n_qubits):
        circ.H(i)
    for layer in range(depth):
        g, b = gammas[layer], betas[layer]
        for (idx, coeff) in diag:
            circ.Rz(g * coeff / np.pi, idx)
        for (i, j, coeff) in off_diag:
            circ.ZZPhase(g * coeff / np.pi, i, j)
        for i in range(n_qubits):
            circ.Rx(2 * b / np.pi, i)
    return circ

In [ ]:
# Run the QAOA Circuit with Guppy & Simulate with Selene

def run_qaoa_circuit(Q, n_qubits, gammas, betas, n_shots=500):
    circ = build_qaoa_pytket(Q, n_qubits, gammas, betas)
    qaoa_gates = guppy.load_pytket("qaoa_gates", circ, use_arrays=True)
    @guppy
    def main() -> None:
        qs = array(qubit() for _ in range(comptime(n_qubits)))
        qaoa_gates(qs)
        result("bits", measure_array(qs))
    shots = main.emulator(n_qubits=n_qubits).with_shots(n_shots).run()
    return shots

In [ ]:
# Converts Raw Selene Output to Numpy Arrays

def shots_to_bitstrings(shots):
    results = []
    for shot in shots:
        bits = shot.as_dict()["bits"]
        results.append(np.array([int(b) for b in bits], dtype=float))
    return results

# Selene returns shots as a list of QsysShot objects. Each shot looks something like:
# pythonQsysShot(entries=[('bits', [True, False, True, False, False])])
# shots_to_bitstrings converts each shot into a numpy array of 0s and 1s:
# python[True, False, True, False, False]  ->  np.array([1.0, 0.0, 1.0, 0.0, 0.0])
# You need numpy arrays so you can compute QUBO energy (x @ Q @ x) and objective value (c_vec @ x). 

In [ ]:
# Check the feasibility of the full output bitstring by verifying that it passed the constraints

def check_feasibility_full(x, A_full, b_full, senses):
    Ax = A_full @ x
    for idx in range(len(b_full)):
        if senses[idx] == 0:
            if abs(Ax[idx] - b_full[idx]) > 1e-6:
                return False
        elif senses[idx] == -1:
            if Ax[idx] > b_full[idx] + 1e-6:
                return False
        elif senses[idx] == 1:
            if Ax[idx] < b_full[idx] - 1e-6:
                return False
    return True

In [ ]:
# QAOA Solver for a single package

# The full QUBO Matrix is a (N*M) by (N*M) huge matrix. Computing that would be very computationally expensive.
# However, we notice that the QUBO Matrix is made of diagonalized blocks, each of which represents the QUBO Matrix for a single package.
# Minimizing the energy with the full QUBO matrix is equivalent to minimizing the energy of the smaller packages.
# These are much more easier to compute.

# This is called block-diagonal decomposition


def solve_package_qaoa(Q_pkg, c_pkg, n_qubits,
                       depth=2, n_shots=500, max_iter=30):
    history = []
    
    def objective(params):
        gammas = params[:depth].tolist()
        betas = params[depth:].tolist()
        shots = run_qaoa_circuit(Q_pkg, n_qubits, gammas, betas, n_shots)
        bitstrings = shots_to_bitstrings(shots)
        energies = [x @ Q_pkg @ x for x in bitstrings]
        avg = np.mean(energies)
        best_idx = np.argmin(energies)
        best_obj = float(c_pkg @ bitstrings[best_idx])
        history.append({"avg_energy": avg, "best_obj": best_obj})
        return avg
    
    init = np.random.uniform(0, np.pi, 2 * depth)
    t0 = time.time()
    opt = minimize(objective, init, method='COBYLA',
                   options={'maxiter': max_iter})
    elapsed = time.time() - t0
    
    # Post-selection with more shots
    best_gammas = opt.x[:depth].tolist()
    best_betas = opt.x[depth:].tolist()
    final_shots = run_qaoa_circuit(Q_pkg, n_qubits, best_gammas, best_betas,
                                   n_shots=2000)
    bitstrings = shots_to_bitstrings(final_shots)
    
    best_obj = -float('inf')
    best_x = None
    for x in bitstrings:
        obj = float(c_pkg @ x)
        if obj > best_obj:
            best_obj = obj
            best_x = x
    
    diag, off_diag = extract_qubo_terms(Q_pkg)
    n_zz = len(off_diag) * depth
    n_1q = (n_qubits + len(diag) + n_qubits) * depth + n_qubits
    
    return {
        "best_x": best_x,
        "best_obj": best_obj,
        "time": elapsed,
        "iterations": len(history),
        "params": opt.x,
        "history": history,
        "n_two_qubit_gates": n_zz,
        "n_one_qubit_gates": n_1q,
        "all_bitstrings": bitstrings,  # keep for combined feasibility
        "total_shots_optimizer": len(history) * n_shots,
        "total_shots_postselect": 2000,
    }

In [8]:
# Workflow:

# Solve classically with CBC to get the baseline optimum
# Build the full QUBO matrix (objective + penalty terms)
# Extract each package's block from the QUBO diagonal
# Run QAOA independently on each package block (small circuits!)
# Combine the per-package solutions into a full solution vector
# Post-selection: sample 2000 combined bitstrings and keep the best one that satisfies ALL constraints (mandatory families, capacity, incompatibility, dependencies)
# Report benchmarks: approximation ratio, feasibility rate, gate counts, timing, solution vectors

def solve_full_qaoa(prob, depth=2, n_shots=500, max_iter=30):
    # Classical baseline
    t_classical = time.time()
    classical = solve_ilp(prob)
    classical_time = time.time() - t_classical
    #
    c_full, A_full, b_full = get_ilp_matrices(prob)
    mod, _ = build_ilp(prob)
    sns = [c.sense for c in mod.constraints.values()]
    lam = 2.0 * np.max(np.abs(c_full)) * prob.M
    Q_full = build_qubo(c_full, A_full, b_full, sns, lam=lam)
    
    N, M = prob.N, prob.M
    
    print(f"Problem: {N} coverages x {M} packages = {N*M} total vars")
    print(f"Circuit size: {N} qubits (per package)")
    print(f"Classical optimum: {classical['objective']:.4f}")
    print(f"  Classical solve time:     {classical_time*1000:.2f}ms")
    print(f"Lambda: {lam:.2f}")
    
    # Solve each package independently
    total_time = 0
    total_qaoa_obj = 0
    total_two_qubit = 0
    total_one_qubit = 0
    total_optimizer_shots = 0
    combined_x = np.zeros(N * M)
    package_results = []
    
    for m in range(M):
        pkg_name = prob.package_names[m] if prob.package_names else f"Package {m}"
        c_pkg = c_full[m*N : (m+1)*N]
        Q_pkg = Q_full[m*N:(m+1)*N, m*N:(m+1)*N]
        classical_pkg_x = classical["solution_vector"][m*N:(m+1)*N]
        classical_pkg_obj = float(c_pkg @ classical_pkg_x)
        
        print(f"\n  Solving {pkg_name} ({N} qubits)...")
        
        pkg_result = solve_package_qaoa(
            Q_pkg, c_pkg, N,
            depth=depth, n_shots=n_shots, max_iter=max_iter
        )
        pkg_result["classical_obj"] = classical_pkg_obj
        pkg_result["package_name"] = pkg_name
        
        ratio = pkg_result["best_obj"] / classical_pkg_obj if classical_pkg_obj > 0 else 0
        print(f"    QAOA: {pkg_result['best_obj']:.2f} | "
              f"Classical: {classical_pkg_obj:.2f} | "
              f"Ratio: {ratio:.4f} | "
              f"Time: {pkg_result['time']:.1f}s")
        
        combined_x[m*N:(m+1)*N] = pkg_result["best_x"]
        total_qaoa_obj += pkg_result["best_obj"]
        total_time += pkg_result["time"]
        total_two_qubit += pkg_result["n_two_qubit_gates"]
        total_one_qubit += pkg_result["n_one_qubit_gates"]
        total_optimizer_shots += pkg_result["total_shots_optimizer"]
        package_results.append(pkg_result)
    
    # ============================================================
    # Combined feasibility check on best solution
    # ============================================================
    combined_feasible = check_feasibility_full(combined_x, A_full, b_full, sns)
    
    # ============================================================
    # Combined post-selection feasibility check
    # Sample across all package combinations from post-selection shots
    # ============================================================
    
    n_postselect = 2000
    feasible_count = 0
    best_feasible_obj = -float('inf')
    best_feasible_x = None
    
    for shot_idx in range(n_postselect):
        # Build combined bitstring from each package's shots
        combined_sample = np.zeros(N * M)
        for m in range(M):
            pkg_bitstrings = package_results[m]["all_bitstrings"]
            # Use modulo in case packages have different shot counts
            idx = shot_idx % len(pkg_bitstrings)
            combined_sample[m*N:(m+1)*N] = pkg_bitstrings[idx]
        
        # Check full feasibility
        if check_feasibility_full(combined_sample, A_full, b_full, sns):
            feasible_count += 1
            obj = float(c_full @ combined_sample)
            if obj > best_feasible_obj:
                best_feasible_obj = obj
                best_feasible_x = combined_sample.copy()
    
    feasibility_rate = feasible_count / n_postselect
    
    # If post-selection found a better feasible solution, use it
    if best_feasible_x is not None and best_feasible_obj > total_qaoa_obj:
        report_obj = best_feasible_obj
        report_x = best_feasible_x
        report_source = "post-selected"
    else:
        report_obj = total_qaoa_obj
        report_x = combined_x
        report_source = "best-energy"

    
    # ============================================================
    # Print all benchmarks — only report feasible solutions
    # ============================================================
    
    print(f"\n{'='*60}")
    print(f"BENCHMARKS ({N} cov x {M} pkg = {N*M} vars)")
    print(f"{'='*60}")
    
    if best_feasible_x is not None:
        report_obj = best_feasible_obj
        report_x = best_feasible_x
        
        print(f"\nSOLUTION QUALITY")
        print(f"  Classical optimum:       {classical['objective']:.4f}")
        print(f"  QAOA best feasible obj:  {report_obj:.4f}")
        print(f"  Approximation ratio:     {report_obj/classical['objective']:.4f}")
        print(f"  Feasible:                True")
        
        print(f"\nFEASIBILITY")
        print(f"  Feasible shots:          {feasible_count} / {n_postselect} "
              f"({feasibility_rate*100:.1f}%)")
        
        print(f"\nTIME TO SOLUTION")
        print(f"  Wall clock time:         {total_time:.2f}s")
        print(f"  Optimizer iterations:    {sum(r['iterations'] for r in package_results)} "
              f"(across {M} packages)")
        print(f"  Optimizer shots:         {total_optimizer_shots}")
        print(f"  Post-selection shots:    {n_postselect * M}")
        print(f"  Total shots:             {total_optimizer_shots + n_postselect * M}")
        
        print(f"\nCIRCUIT METRICS")
        print(f"  Qubits per circuit:      {N}")
        print(f"  QAOA depth (p):          {depth}")
        print(f"  Number of circuits:      {M} (one per package)")
        print(f"  2-qubit gates (per pkg): {package_results[0]['n_two_qubit_gates']}")
        print(f"  1-qubit gates (per pkg): {package_results[0]['n_one_qubit_gates']}")
        print(f"  2-qubit gates (total):   {total_two_qubit}")
        print(f"  1-qubit gates (total):   {total_one_qubit}")
        
        print(f"\nSOLUTION VECTORS")
        print(f"  QAOA (feasible): {report_x.astype(int).tolist()}")
        print(f"  Classical:       {classical['solution_vector'].tolist()}")
        
        print(f"\nPACKAGE BREAKDOWN")
        for m in range(M):
            pkg_name = prob.package_names[m] if prob.package_names else f"Package {m}"
            qaoa_sel = [prob.coverages[i].name for i in range(N) if report_x[i+m*N] == 1]
            clas_sel = [prob.coverages[i].name for i in range(N) 
                        if classical['solution_vector'][i+m*N] == 1]
            print(f"  {pkg_name}:")
            print(f"    QAOA:      {qaoa_sel}")
            print(f"    Classical: {clas_sel}")
    else:
        print(f"\n  NO FEASIBLE SOLUTION FOUND")
        print(f"  Feasible shots: 0 / {n_postselect} (0.0%)")
        print(f"  Try: increase lambda, depth, or shots")
    
    return {
        "qaoa_obj": best_feasible_obj if best_feasible_x is not None else None,
        "classical_obj": classical["objective"],
        "classical_time": classical_time,
        "ratio": best_feasible_obj / classical["objective"] if best_feasible_x is not None else 0,
        "feasible": best_feasible_x is not None,
        "feasibility_rate": feasibility_rate,
        "time": total_time,
        "circuit_qubits": N,
        "n_packages": M,
        "total_vars": N * M,
        "depth": depth,
        "two_qubit_gates_per_pkg": package_results[0]["n_two_qubit_gates"],
        "one_qubit_gates_per_pkg": package_results[0]["n_one_qubit_gates"],
        "two_qubit_gates_total": total_two_qubit,
        "one_qubit_gates_total": total_one_qubit,
        "total_shots": total_optimizer_shots + n_postselect * M,
        "optimizer_iterations": sum(r["iterations"] for r in package_results),
        "combined_x": best_feasible_x if best_feasible_x is not None else combined_x,
    }

In [ ]:
# ============================================================
# Run all scaling instances
# ============================================================
sizes = [
    (5, 2),
    (7, 3),
    (10, 2),
    (10, 5),
    (20, 10),
]

all_scaling = []
for n_cov, n_pkg in sizes:
    print(f"\n{'='*60}")
    print(f"INSTANCE: {n_cov} x {n_pkg} = {n_cov*n_pkg} vars")
    print(f"{'='*60}")
    
    prob = subsample_problem(ltm_full, n_coverages=n_cov, n_packages=n_pkg)
    result_data = solve_full_qaoa(prob, depth=2, n_shots=500, max_iter=30)
    all_scaling.append(result_data)

# ============================================================
# Final scaling table
# ============================================================
print(f"\n{'='*70}")
print(f"SCALING SUMMARY TABLE")
print(f"{'='*70}")
print(f"{'N':>3} {'M':>3} {'vars':>5} {'qubits':>7} {'QAOA':>10} "
      f"{'Classical':>10} {'Ratio':>7} {'Feas%':>6} {'2Q gates':>8} "
      f"{'Shots':>8} {'Time':>6}")
print(f"{'-'*70}")
for r in all_scaling:
    feas_pct = r['feasibility_rate'] * 100
    print(f"{r['circuit_qubits']:>3} {r['n_packages']:>3} "
          f"{r['total_vars']:>5} {r['circuit_qubits']:>7} "
          f"{r['qaoa_obj']:>10.2f} {r['classical_obj']:>10.2f} "
          f"{r['ratio']:>7.4f} {feas_pct:>5.1f}% "
          f"{r['two_qubit_gates_total']:>8} "
          f"{r['total_shots']:>8} {r['time']:>6.1f}")


INSTANCE: 5 x 2 = 10 vars
Problem: 5 coverages x 2 packages = 10 total vars
Circuit size: 5 qubits (per package)
Classical optimum: 710.4906
  Classical solve time:     35.30ms
Lambda: 892.80

  Solving Young Driver Budget (5 qubits)...
    QAOA: 507.31 | Classical: 334.40 | Ratio: 1.5171 | Time: 77.7s

  Solving Young Family Starter (5 qubits)...
    QAOA: 580.19 | Classical: 376.09 | Ratio: 1.5427 | Time: 80.2s

BENCHMARKS (5 cov x 2 pkg = 10 vars)

SOLUTION QUALITY
  Classical optimum:       710.4906
  QAOA best feasible obj:  710.4906
  Approximation ratio:     1.0000
  Feasible:                True

FEASIBILITY
  Feasible shots:          155 / 2000 (7.8%)

TIME TO SOLUTION
  Wall clock time:         157.93s
  Optimizer iterations:    60 (across 2 packages)
  Optimizer shots:         30000
  Post-selection shots:    4000
  Total shots:             34000

CIRCUIT METRICS
  Qubits per circuit:      5
  QAOA depth (p):          2
  Number of circuits:      2 (one per package)
  2-qub